<a href="https://colab.research.google.com/github/huimouqianxiao123/Peft-Qwen2.5-/blob/main/%E5%8A%A0%E8%BD%BD%E6%A8%A1%E5%9E%8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 安装依赖
!pip install -U transformers accelerate bitsandbytes huggingface_hub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 2. 下载模型到 Google Drive (云盘) 目录
from huggingface_hub import snapshot_download

# 注意：去掉了多余的空格，保持官方的仓库名格式
model_id = "Qwen/Qwen2.5-7B-Instruct"

# 将 local_dir 的路径改为你的云盘路径 (我们在 MyDrive 下建一个 models 文件夹专门放它)
local_dir = snapshot_download(
    repo_id=model_id,
    local_dir="/content/drive/MyDrive/models/qwen2.5-7b-instruct",
    local_dir_use_symlinks=False
)

print("模型已成功下载到云盘路径：", local_dir)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

模型已成功下载到云盘路径： /content/drive/MyDrive/models/qwen2.5-7b-instruct


In [ ]:
!ls -lh /content/qwen2.5-7b-instruct

total 15G
-rw-r--r-- 1 root root  663 Jun 20 09:00 config.json
-rw-r--r-- 1 root root  243 Jun 20 09:00 generation_config.json
-rw-r--r-- 1 root root  12K Jun 20 09:00 LICENSE
-rw-r--r-- 1 root root 1.6M Jun 20 09:00 merges.txt
-rw-r--r-- 1 root root 3.7G Jun 20 09:04 model-00001-of-00004.safetensors
-rw-r--r-- 1 root root 3.6G Jun 20 09:03 model-00002-of-00004.safetensors
-rw-r--r-- 1 root root 3.6G Jun 20 09:03 model-00003-of-00004.safetensors
-rw-r--r-- 1 root root 3.4G Jun 20 09:03 model-00004-of-00004.safetensors
-rw-r--r-- 1 root root  28K Jun 20 09:00 model.safetensors.index.json
-rw-r--r-- 1 root root 6.1K Jun 20 09:00 README.md
-rw-r--r-- 1 root root 7.2K Jun 20 09:00 tokenizer_config.json
-rw-r--r-- 1 root root 6.8M Jun 20 09:00 tokenizer.json
-rw-r--r-- 1 root root 2.7M Jun 20 09:00 vocab.json


In [ ]:
!find /content/qwen2.5-7b-instruct -name "*.safetensors"

/content/qwen2.5-7b-instruct/model-00002-of-00004.safetensors
/content/qwen2.5-7b-instruct/model-00003-of-00004.safetensors
/content/qwen2.5-7b-instruct/model-00004-of-00004.safetensors
/content/qwen2.5-7b-instruct/model-00001-of-00004.safetensors


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
model_path = "/content/drive/MyDrive/models/qwen2.5-7b-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
prompt = "介绍一下Transformer模型"

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256
)

print(
    tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )
)

Transformer模型是近年来在自然语言处理领域中非常流行的一种神经网络架构，它由Google的研究人员在2017年提出。Transformer模型主要通过自注意力机制（Self-Attention Mechanism）来捕捉输入序列中的长距离依赖关系，从而显著提高了机器翻译等任务的性能。

### 主要特点

1. **自注意力机制**：这是Transformer最核心的部分。自注意力机制允许模型直接关注输入序列中的重要部分，而不仅仅是顺序信息。这意味着它可以并行处理序列中的不同位置，极大地提高了计算效率。

2. **无递归结构**：与传统的循环神经网络（RNNs）不同，Transformer没有使用递归或循环结构，而是完全基于注意力机制来建模序列之间的依赖关系。这使得Transformer能够更容易地扩展到更长的序列。

3. **可扩展性**：由于其并行化的特点，Transformer非常适合现代大规模分布式计算环境。此外，通过增加模型的层数和隐藏单元的数量，可以进一步提高模型的性能。

4. **注意力机制的优势**：除了上述优点外，自注意力机制还能帮助模型更好地理解上下文信息，这对于处理自然语言任务尤为重要。

### 应用

- **机器翻译
